In [1]:
#!/usr/bin/env python3
"""
Extract a balanced test set from aug_editing_plan.json.

This script samples data evenly across all safety principles.
"""

import json
import os
import random
import argparse
from collections import defaultdict
from typing import List, Dict, Any
import shutil

In [7]:
def extract_principle_id(safety_principle_text: str) -> int:
    """Extract principle ID from safety principle text."""
    if not safety_principle_text:
        return None
    try:
        # Extract the number before the first dot
        pid = int(safety_principle_text.strip().split('.')[0])
        return pid
    except (ValueError, IndexError):
        return None


def group_by_principle(data: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    """Group data samples by their principle ID."""
    grouped = defaultdict(list)
    skipped = 0

    for item in data:
        principle = item.get('safety_risk', {}).get('safety_principle', '')
        pid = extract_principle_id(principle)

        if pid is None:
            skipped += 1
            continue

        grouped[pid].append(item)

    if skipped > 0:
        print(f"Warning: Skipped {skipped} samples with invalid principle format")

    return grouped


def extract_balanced_samples(
    grouped: Dict[int, List[Dict[str, Any]]],
    total_samples: int = 600,
    seed: int = 42
) -> List[Dict[str, Any]]:
    """
    Extract samples evenly distributed across principles.

    For each principle, we try to take `total_samples / num_principles` samples.
    If a principle has fewer samples than required, we take all available samples.
    """
    random.seed(seed)

    num_principles = len(grouped)
    samples_per_principle = total_samples // num_principles

    print(f"Target: {total_samples} samples across {num_principles} principles")
    print(f"Ideal samples per principle: {samples_per_principle}\n")

    selected_samples = []
    total_selected = 0

    for pid in sorted(grouped.keys()):
        available = grouped[pid]
        num_to_select = min(samples_per_principle, len(available))

        # Randomly sample from this principle
        sampled = random.sample(available, num_to_select)
        for i in range(len(sampled)):
            sampled[i].pop('_replacement_meta')
        selected_samples.extend(sampled)

        print(f"Principle {pid:2d}: selected {num_to_select:3d} / {len(available):3d} available")

    total_selected = len(selected_samples)
    print(f"\nTotal selected: {total_selected} samples")

    if total_selected < total_samples:
        print(f"Note: Could only select {total_selected} samples (target was {total_samples})")
        print("      due to limited data in some principles.")

    return selected_samples

In [8]:
input = 'data/action_triggered/aug_editing_plan.json'
output = 'data/test/action_triggered/annotation_data.json'
total_samples = 300
seed = 42

# Load input data
print(f"Loading data from {input}...")
with open(input, 'r') as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples\n")

# Group by principle
grouped = group_by_principle(data)

# Extract balanced samples
selected_samples = extract_balanced_samples(
    grouped,
    total_samples=total_samples,
    seed=seed
)

# Save output
os.makedirs(os.path.dirname(output), exist_ok=True)
with open(output, 'w') as f:
    json.dump(selected_samples, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(selected_samples)} samples to {output}")

Loading data from data/action_triggered/aug_editing_plan.json...
Loaded 6220 samples

Target: 600 samples across 15 principles
Ideal samples per principle: 40

Principle  1: selected  40 / 524 available
Principle  2: selected  40 / 530 available
Principle  3: selected  40 / 510 available
Principle  4: selected  40 / 514 available
Principle  5: selected  40 / 519 available
Principle  6: selected  40 / 523 available
Principle  7: selected  40 /  94 available
Principle  8: selected  40 /  41 available
Principle  9: selected  40 / 181 available
Principle 10: selected  40 / 206 available
Principle 11: selected  40 / 516 available
Principle 12: selected  40 / 516 available
Principle 13: selected  40 / 510 available
Principle 14: selected  40 / 512 available
Principle 15: selected  40 / 523 available

Total selected: 600 samples

Saved 600 samples to data/test/action_triggered/annotation_data.json


In [10]:
# Copy edit images
annotation_file = 'data/test/action_triggered/editing_info.json'
output_dir = 'data/test/action_triggered/edit_image'

# Load annotation data
with open(annotation_file, 'r') as f:
    data = json.load(f)

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Copy images
copied = 0
skipped = 0

for i in range(len(data)):
    item = data[i]
    edit_image_path = item.get('safety_risk', {}).get('edit_image_path', '')
    if not edit_image_path:
        skipped += 1
        continue
    
    source_path = edit_image_path
    filename = os.path.basename(source_path)
    dest_path = os.path.join(output_dir, filename)
    
    if os.path.exists(source_path):
        shutil.copy2(source_path, dest_path)
        copied += 1
        # update edit_image_path
        data[i]['safety_risk']['edit_image_path']=os.path.join("data/test/action_triggered/edit_image", filename)
    else:
        skipped += 1
        print(f"Skipped (not found): {source_path}")

print(f"\nCopied {copied} images to {output_dir}")
print(f"Skipped {skipped} images")
with open("data/test/action_triggered/annotation_info.json", "w") as f:
    json.dump(data, f, indent=2)

Skipped (not found): data/action_triggered/edit_image/furniture_store/0000083__0.png
Skipped (not found): data/action_triggered/edit_image/office/0000061_3__0.png
Skipped (not found): data/action_triggered/edit_image/lecture_theatre/0000110_1__0.png
Skipped (not found): data/action_triggered/edit_image/living_room/img_0283__0.png
Skipped (not found): data/action_triggered/edit_image/recreation_room/0000184__0.png
Skipped (not found): data/action_triggered/edit_image/bedroom/img_0686__0.png
Skipped (not found): data/action_triggered/edit_image/kitchen/img_0748__0.png
Skipped (not found): data/action_triggered/edit_image/home_office/NYU0385__0.png
Skipped (not found): data/action_triggered/edit_image/furniture_store/0000033_14__0.png
Skipped (not found): data/action_triggered/edit_image/classroom/NYU0321__0.png
Skipped (not found): data/action_triggered/edit_image/dining_room/NYU1391__0.png
Skipped (not found): data/action_triggered/edit_image/living_room/NYU1317__0.png
Skipped (not foun